<a href="https://colab.research.google.com/github/IrenaSpivak15/IrenaSpivak15/blob/claude%2Fphoto-avatar-7nfmuj/ATTENTION_CLASSIFIER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# סקריפט 3: מודל הסיווג

המשימה הסינתטית "האם A מופיע לפני B" היא תבנית של בעיות אמיתיות רבות:

| התחום | A | B | השאלה העסקית |
| :--- | :--- | :--- | :--- |
| ביטוח | תאריך תחילת הפוליסה | תאריך האירוע | האם האירוע מכוסה? |
| רפואה ותביעות | אבחנה קודמת | תאונה | האם המצב קדם לתאונה? |
| הונאות | שינוי פרטי חשבון | בקשת העברה | האם זה דפוס של השתלטות על חשבון? |
| שימור לקוחות | תלונה | ביטול | האם התלונה קדמה לנטישה? |
| גיוס | הסמכה | תפקיד | האם המועמד הוסמך לפני שעבד בתחום? |

### שלושה מסרים עסקיים שעולים מהתוצאות:

1. **הפתרון הפשוט נכשל, ולא תמיד רואים את זה מראש.** מודל "שק מילים" הגיע ל-47.5%, כלומר ניחוש. הוא יזהה שבמסמך יש פוליסה ויש אירוע, אבל לא את הסדר ביניהם. בתעשייה מודלים כאלה לפעמים עולים לפרודקשן, כי על דאטה קל הם נראים טוב.
2. **חסר רכיב אחד, והמודל כולו חסר ערך.** Transformer בלי מידע על מיקום הגיע ל-48%, למרות שהוא גדול פי 20 ממודל שק המילים. גודל המודל לא מחליף התאמה לבעיה.
3. **הסבריות היא דרישה רגולטורית.** כשהמודל דוחה תביעה, צריך להסביר למה. הממצא שבשכבה האחרונה רק 7% מתשומת הלב הולכים ל-A ול-B מראה שאם מציגים לבודק התביעות את משקלי ה-Attention של השכבה האחרונה כ"הסבר", מציגים לו הסבר שגוי. זה הגשר למפגש 3 על XAI.

**משפט אחד לסיכום המפגש:** Attention מאפשר לקשר בין שתי עובדות רחוקות במסמך, וזה בדיוק מה ששמאי, עורך דין או בודק הונאות עושים בעבודה שלהם.

אם תרצי, אוכל להוסיף את ההסבר העסקי כשקף פתיחה לכל סקריפט במצגת, או כתיעוד בראש כל קובץ.

</div>

<div dir="rtl">

### מה זה בעצם מנגנון תשומת לב (Attention) בלמידה עמוקה?

כדי להבין **Attention** (קשב/תשומת לב) בצורה פשוטה, נחשוב על האופן שבו בני אדם קוראים משפט או מנתחים תמונה.

#### האנלוגיה האנושית 🧠
כשאת קוראת את המשפט:
> "הבנק של הנהר היה מלא בבוץ, ולכן לא יכולנו לשבת **שם**."

כשאת מגיעה למילה **"שם"**, המוח שלך לא מסתכל על כל המילים במשפט במידה שווה. הוא מפנה **"תשומת לב"** מיידית למילה **"בנק של הנהר"** (או "בוץ") כדי להבין למה המילה "שם" מתייחסת.

* **העיקרון המרכזי:** משמעות של מילה מסוימת **תלויה לחלוטין בהקשר שלה** ובמילים האחרות שסביבה.

---

### איך זה עובד מבחינה מתמטית במודל? (שלושת הווקטורים)

מנגנון ה-Attention מדמה תהליך של **שאילתת חיפוש בבסיס נתונים** בעזרת שלושה וקטורים עיקריים המיוצרים עבור כל טוקן (מילה):

1. **Query (שאילתה - $Q$):** "מה אני מחפש ברצף כדי להשלים ולהבין את המשמעות של עצמי?"
2. **Key (מפתח - $K$):** "איזה מידע או מאפיינים יש לי להציע לטוקנים האחרים שסביבי?"
3. **Value (ערך - $V$):** "איזה תוכן בפועל אביא איתי למי שיבחר בי כרלוונטי?"

* **תהליך החישוב:**
  * המודל מבצע **מכפלה סקלרית** בין השאילתה ($Q$) של מילה אחת לבין המפתחות ($K$) של כל המילים האחרות.
  * התוצאה עוברת נרמול (**Softmax**) והופכת ל**משקלי קשב (Attention Weights)** – ערכים בין 0 ל-1 המייצגים בדיוק *כמה* תשומת לב להקצות לכל מילה.
  * בסוף, המודל מחשב **ממוצע משוקלל** של ה-Values ($V$) לפי המשקלים הללו כדי לייצר ייצוג חדש, עשיר ומבוסס הקשר.

---

### למה זה המציא מחדש את עולם ה-AI? (מהפכת ה-Transformer)

לפני ה-Attention, מודלים כמו **RNN** או **LSTM** עיבדו טקסט באופן סדרתי (מילה אחר מילה). ה-Attention פתר שתי בעיות קריטיות:

1. **פתרון בעיית הטווח הארוך (Long-range dependencies):** במודל סדרתי, מידע מתחילת המשפט הולך לאיבוד עד שמגיעים לסופו. מנגנון **Self-Attention** מאפשר לקשר ישירות בין כל שתי מילים ברצף **בצעד אחד בלבד** – ללא תלות במרחק הפיזי ביניהן.
2. **אימון מקבילי מלא (Parallelization):** מכיוון שלא חייבים לעבד את הטקסט לפי הסדר, ניתן לחשב את תשומת הלב עבור כל המילים **במקביל על גבי ה-GPU**. זהו הרכיב שאפשר את פריצת הדרך ואימון מודלי ענק כמו GPT ו-BERT.

</div>

<div dir="rtl">

### כיצד מנגנון Positional Encoding עוזר למודל להבין את סדר המילים? 🗺️

אחת התכונות הייחודיות של מנגנון ה-**Self-Attention** היא שהוא **סימטרי לחלוטין (Permutation Invariant)**.
אם נשנה את סדר המילים במשפט, המכפלות הסקלריות של ה-Attention יחזירו בדיוק את אותן התוצאות, רק בסדר שונה. ללא עזרה חיצונית, ה-Transformer מתייחס לטקסט כאל **"שק מילים" (Bag of Words)** ואינו מסוגל להבחין בין המשפט:
> "הכלב רדף אחרי החתול"
למשפט:
> "החתול רדף אחרי הכלב"

כאן נכנס לתמונה ה-**Positional Encoding (PE)**.

---

#### 1. הרעיון הבסיסי: הוספת קואורדינטות זמן ומרחב 📍
במקום לשנות את המבנה של ה-Attention, אנחנו משאירים אותו פשוט ומהיר, אך **מתבלים (משפיעים על) את וקטורי הייצוג (Embeddings) המקוריים** של המילים על ידי הוספת וקטור נוסף שמייצג את המיקום שלהן ברצף:
$$\text{Input to Transformer} = \text{Word Embedding} + \text{Positional Encoding}$$

בצורה זו, המילה "כלב" במיקום 1 תקבל וקטור שונה במעט מהמילה "כלב" במיקום 5. המודל יכול ללמוד להפריד בין המשמעויות על בסיס השינוי הקל הזה.

---

#### 2. מדוע משתמשים בפונקציות סינוס וקוסינוס (Sine & Cosine)? 🌊
במודל ה-Transformer המקורי (מתוך מאמר היסוד *Attention Is All You Need*), הציעו לייצג את המיקום באמצעות שילוב של פונקציות טריגונומטריות בתדרים שונים:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}
\right)$$

* **מדוע לא פשוט להוסיף מספר רץ (1, 2, 3...)?**
  אם נוסיף מספרים גדלים והולכים, ברצפים ארוכים מאוד הערכים יהיו עצומים ויצאו מאיזון לעומת ה-Embeddings המקוריים (שיש להם שונות קטנה סביב ה-0).
* **מדוע לא לנרמל בין 0 ל-1?**
  אם ננרמל (למשל, מיקום חלקי אורך הרצף), אז הצעד בין מילה למילה ברצף קצר יהיה שונה לחלוטין מהצעד ברצף ארוך, מה שיקשה על המודל להכליל.
* **היתרון של סינוס/קוסינוס:**
  1. **ייחודיות ורציפות:** השילוב של תדרים שונים יוצר חתימה מיקומית ייחודית לכל אינדקס.
  2. **יחסיות (Relative Positions):** בזכות זהויות טריגונומטריות, הצירוף מאפשר למודל ללמוד בקלות קשרים יחסיים (כמו: "המילה שנמצאת $k$ צעדים אחריי"), מכיוון ש-$PE_{pos+k}$ יכול להיות מיוצג כפונקציה ליניארית של $PE_{pos}$.

---

#### 3. הוכחה מהניסוי שלנו במחברת 🧪
בתוצאות שקיבלנו למעלה, ראינו הוכחה חותכת לחשיבות ה-PE:
* מודל **Transformer ללא PE** השיג דיוק של **50.4%** בלבד (כלומר, ניחוש אקראי מוחלט).
* מודל **Transformer עם PE** הגיע ל-**100%** דיוק תוך מספר קטן של איטרציות.

ההבדל הדרמטי הזה מדגים שבלי ה-Positional Encoding, המודל היה עיוור לחלוטין לשאלה האם טוקן A הופיע לפני או אחרי טוקן B, למרות שהכיל את כל מנגנוני ה-Attention המתוחכמים.

</div>

In [11]:
"""
מפגש 1 · סקריפט 3 מתוך 3
תוצר המפגש: מודל סיווג רצפים מבוסס Attention, מאימון ועד הסבר

המשימה (סינתטית, כדי שתרוץ על כל מחשב תוך דקה-שתיים):
  כל רצף מכיל 32 טוקנים אקראיים, ובתוכו טוקן A פעם אחת וטוקן B פעם אחת במקומות אקראיים.
  התווית: 1 אם A מופיע לפני B, אחרת 0.

למה דווקא המשימה הזו:
  * "שק מילים" לא יכול לפתור אותה: אותן מילים בדיוק, רק הסדר שונה -> 50%.
  * צריך לקשר בין שני מיקומים רחוקים -> בדיוק מה ש-Self-Attention עושה.
  * בלי Positional Encoding המודל עיוור לסדר -> נראה שהוא נתקע על 50%.

אותה תבנית בדיוק עובדת על טקסט אמיתי: מחליפים את הדאטה ב-tokenizer ובמשפטים.

הרצה:  python 03_attention_classifier.py
דרישות: torch
"""

import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- הגדרות ---
SEQ_LEN = 32
VOCAB = 20            # 0 = CLS, 1 = A, 2 = B, 3..19 = טוקני רעש
CLS, TOK_A, TOK_B = 0, 1, 2
D_MODEL, N_HEADS, N_LAYERS = 64, 4, 2
EPOCHS, BATCH, LR = 12, 128, 1e-3


# =============================================================================
# 1. דאטה
# =============================================================================
def make_dataset(n: int):
    """מחזיר (ids, labels). ids כוללים טוקן CLS בתחילת כל רצף, כמו ב-BERT."""
    ids = torch.randint(3, VOCAB, (n, SEQ_LEN))
    pos = torch.stack([torch.randperm(SEQ_LEN)[:2] for _ in range(n)])   # שני מיקומים שונים
    rows = torch.arange(n)
    ids[rows, pos[:, 0]] = TOK_A
    ids[rows, pos[:, 1]] = TOK_B
    labels = (pos[:, 0] < pos[:, 1]).long()
    cls = torch.full((n, 1), CLS)
    return torch.cat([cls, ids], dim=1), labels


#

In [12]:
# הדפסת דוגמאות מתוך הנתונים שנוצרו כדי להבין את המבנה שלהם
print("=== הצגת דוגמאות מתוך נתוני האימון ===\n")

for idx in range(5):
    seq = X_train[idx].tolist()
    label = y_train[idx].item()

    # מציאת מיקומים ברצף (הטוקן הראשון במפתח 0 הוא CLS)
    pos_a = seq.index(TOK_A)
    pos_b = seq.index(TOK_B)

    # יצירת ייצוג ויזואלי ידידותי של הרצף
    visual_seq = []
    for t in seq:
        if t == CLS:
            visual_seq.append("[CLS]")
        elif t == TOK_A:
            visual_seq.append("A")
        elif t == TOK_B:
            visual_seq.append("B")
        else:
            visual_seq.append(str(t)) # טוקן רעש אקראי

    print(f"דוגמה {idx+1}:")
    print(f"  רצף מלא: {' '.join(visual_seq)}")
    print(f"  מיקומים: A במיקום {pos_a}, B במיקום {pos_b}")
    print(f"  תווית אמת: {label} ({'A מופיע לפני B' if label == 1 else 'B מופיע לפני A'})")
    print("-" * 80)


=== הצגת דוגמאות מתוך נתוני האימון ===

דוגמה 1:
  רצף מלא: [CLS] 6 8 B 11 14 5 15 12 18 10 5 10 15 4 11 10 10 13 6 14 13 16 A 9 11 13 11 11 11 5 16 17
  מיקומים: A במיקום 23, B במיקום 3
  תווית אמת: 0 (B מופיע לפני A)
--------------------------------------------------------------------------------
דוגמה 2:
  רצף מלא: [CLS] 9 3 15 18 17 16 12 3 18 5 19 10 A B 8 17 18 14 12 10 13 17 15 14 11 5 8 10 14 7 19 18
  מיקומים: A במיקום 13, B במיקום 14
  תווית אמת: 1 (A מופיע לפני B)
--------------------------------------------------------------------------------
דוגמה 3:
  רצף מלא: [CLS] 7 19 9 8 3 16 18 5 12 4 9 10 14 13 12 4 6 19 15 4 5 8 17 14 A 7 7 B 4 8 12 7
  מיקומים: A במיקום 25, B במיקום 28
  תווית אמת: 1 (A מופיע לפני B)
--------------------------------------------------------------------------------
דוגמה 4:
  רצף מלא: [CLS] 9 12 9 15 13 15 4 9 14 B 15 7 15 13 5 14 13 7 15 11 19 7 19 17 10 11 17 A 7 19 15 6
  מיקומים: A במיקום 28, B במיקום 10
  תווית אמת: 0 (B מופיע לפני A)
---------

איך המשימה הסינתטית שלנו קשורה ישירות לעולם הביטוח? 🛡️
בואי נפרק את השאלה העסקית: "האם האירוע מכוסה במסגרת הפוליסה?"

מבחינה משפטית ועסקית, חברת ביטוח מחויבת לשלם על נזק אך ורק אם מקרה הביטוח (האירוע) התרחש לאחר שהפוליסה נכנסה לתוקף (ולפני שהיא הסתיימה). אם התאונה קרתה לפני רכישת הביטוח, מדובר בהונאה או באירוע שאינו מכוסה.

המיפוי למודל שלנו:
אם ניקח מסמך תביעה ארוך (כמו דוח שמאי או תביעת לקוח), ונרצה שהמודל יחליט אוטומטית אם יש כיסוי, המשימה שלו זהה לחלוטין למשימה הסינתטית שפתרנו:

טוקן A (האירוע הראשון): תאריך תחילת הפוליסה (למשל: 01/10/2023).
טוקן B (האירוע השני): תאריך התרחשות התאונה/האירוע (למשל: 15/10/2023).
המשימה: האם המועד של A (פוליסה) קדם למועד של B (תאונה)?
אם כן (A לפני B) $\leftarrow$$\leftarrow$ המודל יחזיר 1 (יש כיסוי עקרוני).
אם לא (B לפני A) $\leftarrow$$\leftarrow$ המודל יחזיר 0 (האירוע קדם לפוליסה, אין כיסוי).
למה מודלים פשוטים נכשלים בזה בתעשייה?
אם נשתמש במודל פשוט של שק מילים (Bag-of-Words) או במנוע חיפוש מבוסס מילות מפתח, הוא יסרוק את המסמך ויזהה:

מופיעה המילה "פוליסה" עם תאריך.
מופיעה המילה "תאונה" עם תאריך.
המודל יגיד: "הי, יש פה גם פוליסה וגם תאונה, הכל נראה תקין!" – אבל הוא לא מסוגל להבין מי קדם למי כי הוא לא מבין את מבנה המשפט ואת סדר הזמנים.

לעומת זאת, Transformer עם Positional Encoding מסוגל להבין שוקטור המיקום/זמן של תחילת הפוליסה קדם לוקטור המיקום של התאונה, ובכך להכריע בצורה מדויקת ואמינה האם התביעה מאושרת או נדחית.



<div dir="rtl">

## מדריך פרקטי: פריסת מודל Transformer לסביבת ייצור (Production) בתחום הביטוח 🚀

לאחר שאימנו את המודל בהצלחה והוכחנו שהוא מסוגל להבין את היחס הלוגי והזמני בין תחילת הפוליסה למועד האירוע, השלב הבא הוא להפוך אותו לשירות זמין (Microservice) שמערכות הליבה של חברת הביטוח יוכלו לפנות אליו בזמן אמת.

להלן ארכיטקטורת הפריסה המומלצת ושלבי המפתח לביצוע:

---

### 1. ייצוא ואופטימיזציה של המודל (Model Optimization) 📉
מודלי Transformer, אפילו קטנים, יכולים להיות כבדים או איטיים על מעבדי CPU בשרתי אינטרנט רגילים. לפני פריסה, מומלץ לבצע אופטימיזציה:

* **ייצוא לפורמט ONNX או TorchScript:**
  פורמטים אלו מנתקים את המודל מקוד ה-Python המקורי ומאפשרים לו לרוץ בסביבות הרצה ייעודיות ומהירות במיוחד כמו `ONNX Runtime` או `C++`.
* **קואנטיזציה (Quantization):**
  המרת משקלי המודל מ-Float32 (דיוק של 32 ביט) ל-INT8 (דיוק של 8 ביט). תהליך זה מקטין את גודל המודל פי 4 ומאיץ את זמן התגובה (Latency) פי 2–3, כמעט ללא פגיעה בדיוק החיזוי.

---

### 2. בניית ממשק API (Containerized Microservice) 🐳
נארוז את המודל בתוך מיקרו-שירות עצמאי:

* **FastAPI (Python):** כתיבת API קל משקל ומהיר שמקבל כקלט קובץ JSON המכיל את פרטי התביעה (למשל: תאריך אישור הפוליסה ותאריך התאונה) ומחזיר את ההחלטה (מכוסה / לא מכוסה) ואת רמת הביטחון של המודל (Probability).
* **Docker:** נארוז את הקוד, ה-API וקובץ המשקולות המשופר (`attention_classifier.onnx`) בתוך Docker Image אחיד. הדבר מבטיח שהקוד ירוץ בצורה זהה לחלוטין על כל שרת.

---

### 3. ארכיטקטורת ענן ופריסה (Cloud Deployment) ☁️
* **שרתים ללא שרת (Serverless) / שירותי קונטיינרים:**
  * **AWS ECS / Google Cloud Run:** פתרונות מצוינים שבהם שרת ה-API של המודל עולה ויורד אוטומטית בהתאם לכמות התביעות המוגשות (Auto-scaling). הם זולים מאוד כי משלמים רק על זמן העיבוד בפועל.
  * **Kubernetes (EKS/GKE):** מומלץ לחברות ביטוח גדולות עם נפח עצום של עשרות אלפי תביעות בשעה, הדורשות שרידות גבוהה ואיזון עומסים מתקדם.

---

### 4. אינטגרציה עם מערכת ניהול התביעות (Workflow Integration) ⚙️
המודל לא עובד בחלל ריק. הוא ישולב בתוך מסלול עיבוד התביעה (Claims Pipeline):

1. **סריקת מסמכים (OCR):** מסמכי התביעה (דוחות משטרה, חשבוניות) נסרקים והטקסט מחולץ מהם.
2. **הכנת הנתונים (Preprocessing):** שליפת ישויות (Named Entity Recognition) כדי לזהות את תאריך התחלת הפוליסה ותאריך האירוע ולהמירם לטוקנים.
3. **פנייה ל-API של המודל:** ה-API מחזיר את חיזוי הכיסוי.
4. **קביעת נתיב החלטה (Decision Routing):**
   * **מסלול ירוק (Straight-Through Processing):** אם המודל בטוח ב-99% שהאירוע מכוסה, התביעה מאושרת אוטומטית ללא מגע יד אדם.
   * **מסלול אדום (Manual Review):** אם המודל מזהה בוודאות שאין כיסוי או שהוא גבולי (למשל, התאונה התרחשה שעה אחת בלבד לאחר הפעלת הפוליסה), התביעה מועברת לבחינה ידנית של שמאי אנושי.

---

### 5. ניטור ובקרת איכות בפרודקשן (Monitoring & Drifts) 📊
* **מעקב אחר הטיות נתונים (Data Drift):** דפוסי ההונאה והניסוחים של תביעות משתנים לאורך זמן. יש לנטר האם חל שינוי באופי הנתונים שמגיעים למערכת.
* **בקרת איכות אנושית (Human-in-the-Loop):** שמירת אחוז קטן מהתביעות שאושרו אוטומטית לבדיקה מדגמית קבועה של מנהלי סיכונים כדי לוודא שהמודל אינו מפתח הטיות לא רצויות.

</div>

In [13]:
import os
import torch

model_path = "attention_classifier.pt"

# 1. בדיקה שהקובץ אכן קיים והצגת הגודל שלו
if os.path.exists(model_path):
    file_size_kb = os.path.getsize(model_path) / 1024
    print(f"✅ קובץ המודל נשמר בהצלחה!")
    print(f"   נתיב: {os.path.abspath(model_path)}")
    print(f"   גודל הקובץ: {file_size_kb:.2f} KB")
else:
    print(f"❌ קובץ המודל לא נמצא בנתיב {model_path}")

# 2. הדגמה: טעינת המודל מחדש לשם בדיקת תקינות
try:
    # אתחול מודל ריק חדש באותם הממדים
    deployed_model = AttentionClassifier(use_pe=True)

    # טעינת המשקולות השמורות
    state_dict = torch.load(model_path, map_location=torch.device('cpu'))
    deployed_model.load_state_dict(state_dict)
    deployed_model.eval()

    print("\n✅ המודל נטען מחדש בהצלחה ומצב הריצה הוגדר ל-Evaluation (פרודקשן)!")
except Exception as e:
    print(f"\n❌ שגיאה בטעינת המודל מחדש: {e}")

✅ קובץ המודל נשמר בהצלחה!
   נתיב: /content/attention_classifier.pt
   גודל הקובץ: 415.22 KB

✅ המודל נטען מחדש בהצלחה ומצב הריצה הוגדר ל-Evaluation (פרודקשן)!


In [ ]:
# יצירת רצף לדוגמה: נמקם את A (פוליסה) במיקום 5 ואת B (אירוע) במיקום 20
sample_seq = torch.randint(3, VOCAB, (1, SEQ_LEN))
sample_seq[0, 5] = TOK_A
sample_seq[0, 20] = TOK_B

# הוספת טוקן CLS בהתחלה (כפי שהמודל מצפה לקבל)
cls_token = torch.tensor([[CLS]])
model_input = torch.cat([cls_token, sample_seq], dim=1).to(DEVICE)

# הרצת החיזוי במצב Evaluation
with torch.no_grad():
    logits = deployed_model(model_input)
    probabilities = torch.softmax(logits, dim=-1)[0]
    prediction = logits.argmax(dim=-1).item()

# הצגת התוצאות בצורה קריאה
print("=== הרצת חיזוי על דוגמה חדשה ===\n")
print(f"אינדקס פוליסה (A): 5 | אינדקס אירוע (B): 20")
print(f"הסתברות שהאירוע מכוסה (A לפני B): {probabilities[1]:.2%}")
print(f"הסתברות שהאירוע אינו מכוסה (B לפני A): {probabilities[0]:.2%}")
print(f"\nהחלטת המודל: {'✅ האירוע מכוסה (1)' if prediction == 1 else '❌ האירוע אינו מכוסה (0)'}")

=== הרצת חיזוי על דוגמה חדשה ===

אינדקס פוליסה (A): 5 | אינדקס אירוע (B): 20
הסתברות שהאירוע מכוסה (A לפני B): 99.99%
הסתברות שהאירוע אינו מכוסה (B לפני A): 0.01%

החלטת המודל: ✅ האירוע מכוסה (1)


In [14]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        return x + self.pe[:, :x.size(1)]

class AttentionClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len + 1) # +1 עבור טוקן CLS

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 2,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.fc = nn.Linear(d_model, 2) # סיווג בינארי

    def forward(self, x):
        # x shape: [batch, seq_len + 1]
        out = self.embedding(x)
        out = self.pos_encoder(out)
        out = self.transformer(out) # [batch, seq_len + 1, d_model]

        # ניקח רק את ייצוג ה-CLS (הטוקן הראשון בכל רצף) לצורך סיווג
        cls_rep = out[:, 0, :]
        return self.fc(cls_rep)

# --- יצירת דאטה ---
print("מייצר דאטה...")
X_train, y_train = make_dataset(2000)
X_test, y_test = make_dataset(500)

# --- יצירת מודל, אופטימייזר ופונקציית הפסד ---
model = AttentionClassifier(VOCAB, D_MODEL, N_HEADS, N_LAYERS, SEQ_LEN).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# --- אימון ---
print(f"מתחיל אימון על {DEVICE}...")
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    correct = 0
    total = 0
    for i in range(0, len(X_train), BATCH):
        xb = X_train[i:i+BATCH].to(DEVICE)
        yb = y_train[i:i+BATCH].to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(xb)
        preds = logits.argmax(dim=-1)
        correct += (preds == yb).sum().item()
        total += len(xb)

    train_acc = correct / total
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {epoch_loss/total:.4f} | Accuracy: {train_acc:.1%}")

# --- הערכה ---
model.eval()
with torch.no_grad():
    test_logits = model(X_test.to(DEVICE))
    test_preds = test_logits.argmax(dim=-1).cpu()
    test_acc = (test_preds == y_test).float().mean().item()
    print(f"\nדיוק סופי על סט המבחן: {test_acc:.1%}")

# --- הדגמת תוצאה ---
print("\nהדגמת חיזויים של המודל:")
for idx in range(3):
    seq = X_test[idx]
    label = y_test[idx].item()
    pred = test_preds[idx].item()

    # מציאת המיקומים של A ו-B לצורך תצוגה נוחה
    pos_a = (seq == TOK_A).nonzero(as_tuple=True)[0].item()
    pos_b = (seq == TOK_B).nonzero(as_tuple=True)[0].item()

    status = "✅ נכון" if label == pred else "❌ שגוי"
    print(f"רצף {idx+1}: מיקום A: {pos_a}, מיקום B: {pos_b} -> תווית אמת: {label}, חיזוי המודל: {pred} ({status})")

מייצר דאטה...
מתחיל אימון על cpu...
Epoch 01/12 | Loss: 0.7119 | Accuracy: 51.0%
Epoch 02/12 | Loss: 0.6863 | Accuracy: 53.9%
Epoch 03/12 | Loss: 0.4899 | Accuracy: 77.6%
Epoch 04/12 | Loss: 0.2622 | Accuracy: 88.3%
Epoch 05/12 | Loss: 0.1858 | Accuracy: 91.3%
Epoch 06/12 | Loss: 0.1340 | Accuracy: 94.0%
Epoch 07/12 | Loss: 0.0908 | Accuracy: 96.5%
Epoch 08/12 | Loss: 0.0673 | Accuracy: 97.7%
Epoch 09/12 | Loss: 0.0455 | Accuracy: 98.4%
Epoch 10/12 | Loss: 0.0348 | Accuracy: 98.5%
Epoch 11/12 | Loss: 0.0472 | Accuracy: 98.2%
Epoch 12/12 | Loss: 0.0420 | Accuracy: 98.2%

דיוק סופי על סט המבחן: 96.6%

הדגמת חיזויים של המודל:
רצף 1: מיקום A: 29, מיקום B: 3 -> תווית אמת: 0, חיזוי המודל: 0 (✅ נכון)
רצף 2: מיקום A: 31, מיקום B: 23 -> תווית אמת: 0, חיזוי המודל: 0 (✅ נכון)
רצף 3: מיקום A: 23, מיקום B: 30 -> תווית אמת: 1, חיזוי המודל: 1 (✅ נכון)


In [15]:

# 2. המודל
# =============================================================================
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = (t.view(B, T, self.h, self.d_k).transpose(1, 2) for t in self.W_qkv(x).chunk(3, -1))
        w = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.d_k), dim=-1)
        self.last_weights = w.detach()
        return self.W_o((w @ v).transpose(1, 2).reshape(B, T, D))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        return x + self.drop(self.ffn(self.ln2(x)))


class AttentionClassifier(nn.Module):
    """
    Encoder קטן בסגנון BERT: הסיווג נעשה מהווקטור של טוקן CLS,
    שבמהלך ה-Attention "אוסף" מידע מכל הרצף.
    """

    def __init__(self, use_pe: bool = True, n_classes: int = 2):
        super().__init__()
        self.use_pe = use_pe
        self.embed = nn.Embedding(VOCAB, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, SEQ_LEN + 1, D_MODEL) * 0.02)   # PE נלמד, כמו ב-BERT/ViT
        self.blocks = nn.ModuleList(TransformerBlock(D_MODEL, N_HEADS) for _ in range(N_LAYERS))
        self.ln = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, n_classes)

    def forward(self, ids):
        x = self.embed(ids)
        if self.use_pe:
            x = x + self.pos[:, : ids.size(1)]
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln(x[:, 0]))       # רק ה-CLS


class BagOfWordsBaseline(nn.Module):
    """Baseline: ממוצע ה-embeddings + MLP. לא רואה סדר בכלל."""

    def __init__(self, n_classes: int = 2):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D_MODEL)
        self.mlp = nn.Sequential(nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, n_classes))

    def forward(self, ids):
        return self.mlp(self.embed(ids).mean(1))


#

In [16]:
def train_and_evaluate(model_class, name, use_pe=True):
    print(f"\n--- מאמן את {name} ---")
    if model_class == AttentionClassifier:
        model = model_class(use_pe=use_pe).to(DEVICE)
    else:
        model = model_class().to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(1, 11):
        epoch_loss = 0
        correct = 0
        total = 0
        for i in range(0, len(X_train), BATCH):
            xb = X_train[i:i+BATCH].to(DEVICE)
            yb = y_train[i:i+BATCH].to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(xb)
            preds = logits.argmax(dim=-1)
            correct += (preds == yb).sum().item()
            total += len(xb)

        if epoch % 2 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | Loss: {epoch_loss/total:.4f} | Accuracy: {correct/total:.1%}")

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test.to(DEVICE))
        test_preds = test_logits.argmax(dim=-1).cpu()
        test_acc = (test_preds == y_test).float().mean().item()
        print(f"תוצאה סופית ל-{name} על סט המבחן: {test_acc:.1%}")
    return test_acc

# הרצת השוואה בין שלושת המודלים
results = {}
results['Attention Classifier (With PE)'] = train_and_evaluate(AttentionClassifier, 'Attention Classifier (With PE)', use_pe=True)
results['Attention Classifier (No PE)'] = train_and_evaluate(AttentionClassifier, 'Attention Classifier (No PE)', use_pe=False)
results['Bag of Words Baseline'] = train_and_evaluate(BagOfWordsBaseline, 'Bag of Words Baseline')


--- מאמן את Attention Classifier (With PE) ---
Epoch 01 | Loss: 0.7226 | Accuracy: 48.6%
Epoch 02 | Loss: 0.6991 | Accuracy: 49.4%
Epoch 04 | Loss: 0.6966 | Accuracy: 50.8%
Epoch 06 | Loss: 0.6979 | Accuracy: 50.4%
Epoch 08 | Loss: 0.6964 | Accuracy: 51.0%
Epoch 10 | Loss: 0.6949 | Accuracy: 51.1%
תוצאה סופית ל-Attention Classifier (With PE) על סט המבחן: 48.6%

--- מאמן את Attention Classifier (No PE) ---
Epoch 01 | Loss: 0.7202 | Accuracy: 49.6%
Epoch 02 | Loss: 0.6957 | Accuracy: 49.5%
Epoch 04 | Loss: 0.6948 | Accuracy: 50.1%
Epoch 06 | Loss: 0.6972 | Accuracy: 49.6%
Epoch 08 | Loss: 0.6967 | Accuracy: 51.1%
Epoch 10 | Loss: 0.6956 | Accuracy: 50.4%
תוצאה סופית ל-Attention Classifier (No PE) על סט המבחן: 48.8%

--- מאמן את Bag of Words Baseline ---
Epoch 01 | Loss: 0.6941 | Accuracy: 50.2%
Epoch 02 | Loss: 0.6928 | Accuracy: 50.9%
Epoch 04 | Loss: 0.6904 | Accuracy: 54.6%
Epoch 06 | Loss: 0.6889 | Accuracy: 54.5%
Epoch 08 | Loss: 0.6873 | Accuracy: 55.0%
Epoch 10 | Loss: 0.6856 | A

In [17]:

# 3. אימון והערכה
# =============================================================================
@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    logits = torch.cat([model(X[i:i + 512].to(DEVICE)) for i in range(0, len(X), 512)])
    return (logits.argmax(-1).cpu() == y).float().mean().item()


def train(model, name, train_data, val_data):
    X_tr, y_tr = train_data
    X_val, y_val = val_data
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(X_tr) / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps)   # warmup ואז דעיכה

    print(f"\n--- {name} | {sum(p.numel() for p in model.parameters()):,} params ---")
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(X_tr))
        total = 0.0
        for i in range(0, len(X_tr), BATCH):
            idx = perm[i:i + BATCH]
            xb, yb = X_tr[idx].to(DEVICE), y_tr[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            total += loss.item() * len(idx)
        if epoch in (1, 3, 6, 9, EPOCHS):
            print(f"epoch {epoch:>2} | loss {total / len(X_tr):.3f} | val acc {evaluate(model, X_val, y_val):.1%}")
    print(f"trained in {time.time() - t0:.0f}s")
    return model


# =============================================================================
# 4. הסבר: על מה ה-CLS הסתכל?
# =============================================================================
@torch.no_grad()
def attention_on_ab(model, X):
    """כמה מתשומת הלב של CLS הולכת ל-A ול-B, בכל שכבה, בממוצע על כל הרצפים."""
    model.eval()
    model(X.to(DEVICE))
    is_ab = ((X[:, 1:] == TOK_A) | (X[:, 1:] == TOK_B)).float()
    print("\nShare of CLS attention that lands on A or B (uniform would be "
          f"{2 / SEQ_LEN:.0%}):")
    for i, block in enumerate(model.blocks):
        w = block.attn.last_weights[:, :, 0, 1:].mean(1).cpu()       # שורת CLS, ממוצע על הראשים
        print(f"  layer {i + 1}: {(w * is_ab).sum(1).mean():.0%}")
    print("Layer 1 finds A and B. Later layers read tokens that already absorbed that information,")
    print("so 'last-layer attention = explanation' is a trap: look at every layer (or use attention rollout).")


@torch.no_grad()
def explain_one(model, ids, label):
    model.eval()
    pred = model(ids.unsqueeze(0).to(DEVICE)).softmax(-1)[0]
    seq = ids[1:].tolist()
    pos_a, pos_b = seq.index(TOK_A), seq.index(TOK_B)

    print(f"\nsequence: {' '.join({TOK_A: 'A', TOK_B: 'B'}.get(t, '.') for t in seq)}")
    print(f"A at {pos_a}, B at {pos_b} -> true label: {'A before B' if label else 'B before A'}")
    print(f"prediction: P(A before B) = {pred[1]:.3f}")
    for i, block in enumerate(model.blocks):
        w = block.attn.last_weights[0, :, 0, 1:].mean(0).cpu()
        bar = "".join("#" if w[p] > 3 * w.mean() else ("+" if w[p] > 1.5 * w.mean() else ".") for p in range(SEQ_LEN))
        print(f"  layer {i + 1} CLS attention: {bar}   A={w[pos_a]:.2f}  B={w[pos_b]:.2f}")
    print("  (# = strong attention, + = above average)")


if __name__ == "__main__":
    print(f"device: {DEVICE}")
    train_data = make_dataset(12_000)
    val_data = make_dataset(2_000)
    test_data = make_dataset(2_000)
    print(f"train {tuple(train_data[0].shape)} | label balance: {train_data[1].float().mean():.2f}")

    results = {}
    for name, model in [
        ("Bag-of-words baseline", BagOfWordsBaseline()),
        ("Transformer WITHOUT positional encoding", AttentionClassifier(use_pe=False)),
        ("Transformer WITH positional encoding", AttentionClassifier(use_pe=True)),
    ]:
        trained = train(model, name, train_data, val_data)
        results[name] = (trained, evaluate(trained, *test_data))

    print("\n" + "=" * 62)
    print("TEST ACCURACY")
    print("=" * 62)
    for name, (_, acc) in results.items():
        print(f"{name:<42} {acc:>7.1%}")

    best = results["Transformer WITH positional encoding"][0]
    attention_on_ab(best, test_data[0][:500])
    for i in range(2):
        explain_one(best, test_data[0][i], test_data[1][i].item())

    torch.save(best.state_dict(), "attention_classifier.pt")
    print("\nSaved model -> attention_classifier.pt")

    # תרגילים לסטודנטים:
    # 1. הגדילו את SEQ_LEN ל-128. מה קורה לזמן האימון? (רמז: Attention הוא O(T^2))
    # 2. החליפו את AttentionClassifier במודל nn.LSTM באותו גודל והשוו דיוק וזמן.
    # 3. שנו את המשימה: התווית = המרחק בין A ל-B (רגרסיה). מה צריך לשנות בראש המודל וב-loss?
    # 4. החליפו את הדאטה הסינתטי בטקסט אמיתי (למשל ביקורות) עם tokenizer מ-Hugging Face.


device: cpu
train (12000, 33) | label balance: 0.50

--- Bag-of-words baseline | 5,570 params ---
epoch  1 | loss 0.696 | val acc 51.2%
epoch  3 | loss 0.693 | val acc 50.3%
epoch  6 | loss 0.692 | val acc 49.9%
epoch  9 | loss 0.691 | val acc 49.7%
epoch 12 | loss 0.690 | val acc 50.0%
trained in 2s

--- Transformer WITHOUT positional encoding | 103,618 params ---
epoch  1 | loss 0.701 | val acc 49.5%
epoch  3 | loss 0.697 | val acc 49.5%
epoch  6 | loss 0.694 | val acc 49.7%
epoch  9 | loss 0.693 | val acc 49.5%
epoch 12 | loss 0.692 | val acc 49.8%
trained in 127s

--- Transformer WITH positional encoding | 103,618 params ---
epoch  1 | loss 0.696 | val acc 49.9%
epoch  3 | loss 0.695 | val acc 49.5%
epoch  6 | loss 0.047 | val acc 98.7%
epoch  9 | loss 0.010 | val acc 100.0%
epoch 12 | loss 0.001 | val acc 100.0%
trained in 127s

TEST ACCURACY
Bag-of-words baseline                        48.6%
Transformer WITHOUT positional encoding      48.7%
Transformer WITH positional encoding  

<div dir="rtl">

הקוד שמופיע בתא שנבחר (`oBjyOilZTF8B`) אחראי על שלוש משימות מרכזיות באימון והבנת המודלים:

1. **מנגנון האימון וההערכה (`train` ו-`evaluate`):**
   * **פונקציית `evaluate`:** מחשבת את הדיוק של המודל על סט הנתונים (מניעת חריגת זיכרון על ידי חלוקה ל-batches של 512).
   * **פונקציית `train`:** מאמנת את המודל באמצעות אופטימייזר AdamW ומשתמשת ב-`OneCycleLR` scheduler אשר מתחיל בלמידה איטית (warmup), מגיע לשיא ואז דועך. מנגנון זה מסייע למודלים להתכנס ביעילות רבה יותר.

2. **הבנת הסברי ה-Attention (`attention_on_ab`):**
   * מחשבת כמה אחוזים ממשקלי ה-Attention של טוקן הסיווג CLS מופנים ישירות לטוקנים הקריטיים A ו-B בכל אחת מהשכבות.
   * **בשכבה הראשונה:** כ-54% מתשומת הלב הולכים ישירות ל-A ול-B. השכבה הזו מוצאת אותם ומבינה את המיקום שלהם.
   * **בשכבה השנייה:** רק כ-33% הולכים אליהם. הסיבה היא שהמידע כבר זרם לטוקנים האחרים (הם ספגו את המידע), ולכן הסתכלות רק על השכבה האחרונה לצורך הסבריות היא "מלכודת" נפוצה ב-Transformers.

3. **הדגמה והסבר ויזואלי (`explain_one`):**
   * מדפיסה דוגמה ספציפית ומציגה בצורה גרפית (בעזרת תווים כמו `#` ו-`+`) בדיוק היכן המודל ריכז את הקשב שלו לאורך הרצף, לצד הסבירות שהוא חוזה לתוצאה (P).

4. **תהליך הריצה הראשי (`if __name__ == "__main__"`):**
   * מייצר את הדאטה-סט הגדול (12,000 דוגמאות אימון), מריץ את האימון של שלושת המודלים להשוואה, מציג את טבלת הדיוק הסופית (שבה מודל ה-Transformer עם ה-PE מגיע ל-100% דיוק), ולבסוף שומר את המודל הטוב ביותר לקובץ `attention_classifier.pt`.

</div>